In [60]:
import pandas as pd
from nltk import word_tokenize, sent_tokenize
from nltk.corpus import stopwords, opinion_lexicon
import string
from nltk.stem import PorterStemmer
from nltk.util import ngrams
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import VarianceThreshold
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from wordcloud import WordCloud
import numpy as np
import networkx as nx

df = pd.read_csv("netflix_reviews.csv")


In [61]:
df.columns

Index(['reviewId', 'userName', 'content', 'score', 'thumbsUpCount',
       'reviewCreatedVersion', 'at', 'appVersion'],
      dtype='str')

In [62]:
df = df.drop(columns = ['reviewId', 'userName', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'appVersion'])
df.columns

Index(['content', 'score'], dtype='str')

In [63]:
# making a sample of 10000 entries from the dataset
sample, _ = train_test_split(df, train_size = 10000, stratify = df['score'], random_state = 42)
sample.reset_index(drop = True, inplace = True)
sample['score'].value_counts()

score
1    3930
5    3173
4    1084
3     946
2     867
Name: count, dtype: int64

In [64]:
len(sample)

10000

In [65]:
# droping null values, removing punctuation, and converting to lowercase
sample = sample.dropna()
sample['content'] = sample['content'].str.replace(r"[^A-Za-z\s]", "", regex=True).str.lower()

In [66]:
len(sample)

9998

In [67]:
# no more null values
sample.isna().sum()

content    0
score      0
dtype: int64

In [68]:
# making a new column which has a list of all words in that row's content column and then dropping the content column
sample['content_tokenized'] = sample['content'].apply(word_tokenize) 
sample = sample.drop(columns = ['content'])

In [69]:
# defining set of stopwords
stop_words = set(stopwords.words('english'))

# defining function to remove stop words
def remove_stop(review):
    filtered = [word for word in review if word not in stop_words]
    return filtered

In [70]:
# removing stop words from content_tokenized column
sample['content_tokenized'] = sample['content_tokenized'].apply(remove_stop)

In [79]:
# making another column in sample with the words from the cleaned content column as strings
def into_string(content):
    return " ".join(content)

sample['content_string'] = sample['content_tokenized'].apply(into_string)

In [80]:
# swapping order of columns for improved readablility
sample = sample[['content_tokenized', 'content_string', 'score']]

In [81]:
sample.head(10)

,content_tokenized,content_string,score
0,"[used, netflix, years, mainly, low, pricing, g...",used netflix years mainly low pricing good sel...,1
1,"[playing, anything, huawei, p, pro, everytime,...",playing anything huawei p pro everytime video ...,1
2,"[new, update, stupid, dad, lives, different, s...",new update stupid dad lives different state us...,2
3,"[nice, app]",nice app,5
4,"[dont, know, download, movie, become, slowi, n...",dont know download movie become slowi need res...,3
5,"[love, neftlix, love, watch, movies, tv, dead,...",love neftlix love watch movies tv dead fun wat...,5
6,[good],good,5
7,"[lets, take, im, using, x, profile, showing, p...",lets take im using x profile showing profile l...,5
8,"[still, semi, regularly, hands, logo, paying, ...",still semi regularly hands logo paying content...,1
9,"[app, open, oneplus, pro, stuck, netflix, load...",app open oneplus pro stuck netflix loading scr...,1
